# ETF-holdings → `holdings_long.csv`

Deze notebook is de **actuele pipeline** voor het inlezen, normaliseren en valideren van ETF-holdings. De oudere, genummerde notebookversies staan in `archive/` en zijn alleen ter referentie.

## Bestanden en mappen

- `data/inbox/`: holdingsbestanden (`.xlsx`) en `etf_master.csv`.
- `data/output/`: de gegenereerde `holdings_long.csv` en `validation_report.csv`.
- De notebook staat in `src/`. De projectmap wordt automatisch gevonden vanuit de huidige werkmap. Als dat niet lukt, kan `HOLDINGS_PROJECT_ROOT` op de projectmap worden gezet.

## Gebruik

Vereist: `pandas`, `numpy` en `openpyxl` (`pip install pandas numpy openpyxl`). Voer de notebookcellen van boven naar beneden uit. De uitgever wordt aan de hand van de werkmapinhoud herkend. Bestanden met validatiestatus `FAIL` worden standaard uitgesloten; zet `HOLDINGS_INCLUDE_FAILED=1` om ze toch op te nemen.

De parseronderdelen ondersteunen Amundi, iShares, SPDR en Fidelity. De validatierapportage geeft per bestand status, gewichtensom, fouten en waarschuwingen.

# fouten in de huidige logica  
## de percentages  
Vermits het doel is te kijken welke holdings, en of sectoren en of landen een te groot gewicht hebben in de totale portefeuille (etf_master.csv) moeten er een veld worden toegevoegd namelijk hoeveel er van elke ETF in portefeuille zitten.  
Met de percentages zijn we rechtstreeks niets.  Deze moeten per ETF en per holding omgezet worden in functie van de actuele prijs en het aantal etf in portefuille.  
### voorstel Copilot  
Read 
Completed with input: {
  "path": "C:\\Users\\Wilfried\\AppData\\Roaming\\Code\\agentSessionData\\db3c06c7-bc92-4523-8b6a-e17efab9d475\\attachments\\f25714dc-1644-4953-b7bb-d7a6558ee2b2\\file_holdings_pipeline.ipynb"
}

Read , lines 1 to 60
Completed with input: {
  "path": "D:\\Projects\\Domains\\assetmanagement\\projects\\ETF\\src\\holdings_pipeline.ipynb",
  "view_range": [
    1,
    60
  ]
}

Read 
Completed with input: {
  "path": "C:\\Users\\Wilfried\\AppData\\Roaming\\Code\\agentSessionData\\db3c06c7-bc92-4523-8b6a-e17efab9d475\\attachments\\6efb5c21-0aad-4282-aa5c-ab7e5584e1b0\\file_holdings_pipeline.ipynb output.json"
}

Je toevoeging benoemt een **echte beperking**, maar “met de percentages zijn we rechtstreeks niets” is te absoluut. De gewichten in de holdingsbestanden zijn wél nuttig: ze geven aan welk deel van een ETF in een onderliggende holding zit. Wat nu ontbreekt, is de koppeling met **jouw positie in elke ETF**.

De berekening is in principe:

- **Waarde ETF-positie** = aantal ETF-eenheden × actuele koers × eventuele valutaomrekening.
- **Waarde van een holding via die ETF** = waarde ETF-positie × holdinggewicht.
- Tel die blootstellingen op over de ETF’s en deel eventueel door de totale portefeuillewaarde om portefeuillepercentages per holding, sector of land te krijgen.

Dat geeft een **geschatte blootstelling in euro’s**, geen exact aantal onderliggende aandelen. Daarvoor heb je ook aantallen of nominale waarden van de onderliggende posities nodig. De schatting hangt bovendien af van hoe recent de holdingsgewichten zijn.

Ik zou de aantallen niet aan `etf_master.csv` toevoegen: dat bestand bevat ETF-kenmerken, zoals ISIN en TER, terwijl aantallen per portefeuille en per peildatum variëren. Een aparte portefeuilleposities-tabel ligt meer voor de hand, met bijvoorbeeld ISIN, aantal, koersdatum en eventueel portefeuillevaluta. Koersen en FX-koersen zijn aparte, tijdsafhankelijke gegevens.

Een preciezere formulering voor de toevoeging zou zijn:

> **Portefeuilleblootstelling**  
> De holdingspercentages geven de verdeling binnen elke ETF, maar niet het gewicht van die ETF in mijn totale portefeuille. Om holdings, sectoren en landen over de hele portefeuille te berekenen, zijn ook mijn aantallen per ETF en de waarde van die posities nodig. Vermenigvuldig de ETF-positiewaarde met het holdinggewicht en tel de blootstellingen uit alle ETF’s bij elkaar op. De uitkomst is een schatting op basis van de rapportagedatum van de holdings.

Zo wordt ook duidelijker dat `etf_master.csv` waarschijnlijk niet de portefeuilleposities zelf hoort te bevatten.  

In [ ]:
import os, re, warnings, datetime as dt
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", message="Workbook contains no default style")
pd.set_option("display.max_colwidth", None, "display.width", 250, "display.max_columns", 30)

# ---- configuratie ----
def resolve_project_root():
    override = os.environ.get("HOLDINGS_PROJECT_ROOT")
    if override:
        root = Path(override).expanduser().resolve()
        if (root / "data" / "inbox" / "etf_master.csv").is_file():
            return root
        raise FileNotFoundError(f"Geen data/inbox/etf_master.csv gevonden onder HOLDINGS_PROJECT_ROOT: {root}")

    cwd = Path.cwd().resolve()
    for parent in (cwd, *cwd.parents):
        if (parent / "data" / "inbox" / "etf_master.csv").is_file():
            return parent
        project = parent / "projects" / "ETF"
        if (project / "data" / "inbox" / "etf_master.csv").is_file():
            return project
    raise FileNotFoundError(
        f"Projectmap niet gevonden vanuit {cwd}; stel HOLDINGS_PROJECT_ROOT in op de map met data/inbox/etf_master.csv."
    )


BASE = resolve_project_root()
INBOX, OUTPUT, MASTER = BASE / "data" / "inbox", BASE / "data" / "output", BASE / "data" / "inbox" / "etf_master.csv"
INCLUDE_FAILED = os.environ.get("HOLDINGS_INCLUDE_FAILED", "0") == "1"   # True = ook FAIL-bestanden wegschrijven

# toegestane afwijking van 100 (procentpunten) op de gewichtensom, per uitgever
WEIGHT_TOL = {"Amundi": 0.5, "iShares": 0.5, "SPDR": 0.5, "Fidelity": 1.0}

SCHEMA = ["snapshot_date", "etf_isin", "etf_naam", "bron_type", "replicatie",
          "holding_isin", "holding_naam", "asset_class", "valuta", "gewicht_pct", "sector", "land"]
EXTRA = ["uitgever", "holding_ticker", "beurs", "fonds_in_fonds", "bronbestand"]   # extra kolommen achteraan

In [ ]:
# ---- hulpfuncties ----
ISIN_RE = re.compile(r"^[A-Z]{2}[A-Z0-9]{9}[0-9]$")
ISIN_FILE_RE = re.compile(r"[A-Z]{2}[A-Z0-9]{9}[0-9]")

MONTHS = {"jan": 1, "feb": 2, "mar": 3, "mrt": 3, "maa": 3, "apr": 4, "may": 5, "mei": 5, "jun": 6, "jul": 7,
          "aug": 8, "sep": 9, "oct": 10, "okt": 10, "nov": 11, "dec": 12}


def clean(v):
    """Lege cel/NaN/lege string -> None, anders getrimde string."""
    if v is None or (not isinstance(v, str) and pd.isna(v)):
        return None
    s = str(v).strip()
    return s or None


def to_float(v):
    """Getal uit cel; '-', leeg -> NaN. Begrijpt ook '2.950,00' (NL) en '12,5 %'."""
    if v is None or (not isinstance(v, str) and pd.isna(v)):
        return np.nan
    if isinstance(v, (int, float, np.integer, np.floating)):
        return float(v)
    s = str(v).strip().replace("%", "").replace("\xa0", "").replace(" ", "")
    if s in ("", "-", "--"):
        return np.nan
    if "," in s:
        s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan


def parse_date(v):
    """ISO, dd/mm/yyyy, '03-Nov-2026' en NL '03/nov/2026' -> datetime.date."""
    if isinstance(v, (dt.datetime, pd.Timestamp)):
        return v.date()
    if isinstance(v, dt.date):
        return v
    s = clean(v)
    if not s:
        return None
    s = s.lower()
    if m := re.search(r"(\d{4})-(\d{2})-(\d{2})", s):
        return dt.date(int(m[1]), int(m[2]), int(m[3]))
    if m := re.search(r"(\d{1,2})/(\d{1,2})/(\d{4})", s):
        return dt.date(int(m[3]), int(m[2]), int(m[1]))
    if m := re.search(r"(\d{1,2})[-/ .]([a-z]{3,})\.?[-/ .](\d{4})", s):
        mon = MONTHS.get(m[2][:3])
        if mon:
            return dt.date(int(m[3]), mon, int(m[1]))
    raise ValueError(f"Datum niet te lezen: {v!r}")


def isin_ok(s):
    """Formaat + Luhn-controlecijfer."""
    if not isinstance(s, str) or not ISIN_RE.match(s):
        return False
    digits = "".join(str(int(c, 36)) for c in s)
    total = 0
    for i, ch in enumerate(reversed(digits)):
        d = int(ch)
        if i % 2 == 1:
            d *= 2
            d = d - 9 if d > 9 else d
        total += d
    return total % 10 == 0


def norm_replication(v):
    s = (clean(v) or "").lower()
    if not s:
        return None
    if "synth" in s or "indirect" in s or "swap" in s:
        return "Synthetic"
    if "phys" in s or "fysiek" in s or "direct" in s:
        return "Physical"
    return clean(v)


def find_header_row(raw, required, scan=60):
    req = {r.lower() for r in required}
    for i in range(min(scan, len(raw))):
        vals = {str(v).strip().lower() for v in raw.iloc[i].tolist() if clean(v)}
        if req <= vals:
            return i
    raise ValueError(f"Headerrij niet gevonden: {required}")


def table_below(raw, h):
    t = raw.iloc[h + 1:].copy()
    t.columns = [clean(c) or f"_c{i}" for i, c in enumerate(raw.iloc[h])]
    return t.reset_index(drop=True)


def has_text(raw, text):
    return raw.astype(str).apply(lambda c: c.str.contains(text, case=False, regex=False)).any().any()


# ---- vertaaltabellen voor de Nederlandstalige iShares-export ----
# Eerste groep is in het voorbeeldbestand gezien; de rest zijn aannames (onbekende labels worden gemeld als WARN).
SECTOR_NL_EN = {
    "Financiële waarden": "Financials", "Luxe-consumentengoederen": "Consumer Discretionary",
    "IT": "Information Technology", "Communicatie": "Communication Services", "Energie": "Energy",
    "Industrie": "Industrials", "Materialen": "Materials",
    "Gezondheidszorg": "Health Care", "Basisconsumptiegoederen": "Consumer Staples",
    "Basis-consumentengoederen": "Consumer Staples", "Nutsbedrijven": "Utilities", "Vastgoed": "Real Estate",
    "Liquide middelen en/of derivaten": "Other", "Overige": "Other",
}
COUNTRY_NL_EN = {
    "Brazilië": "Brazil", "China": "China", "Duitsland": "Germany", "Hongarije": "Hungary", "Ierland": "Ireland",
    "India": "India", "Koeweit": "Kuwait", "Mexico": "Mexico", "Polen": "Poland", "Saoedi-Arabië": "Saudi Arabia",
    "Taiwan": "Taiwan", "Zuid-Afrika": "South Africa", "Zuid-Korea": "South Korea",
    "Chili": "Chile", "Colombia": "Colombia", "Egypte": "Egypt", "Filipijnen": "Philippines",
    "Griekenland": "Greece", "Hongkong": "Hong Kong", "Indonesië": "Indonesia", "Maleisië": "Malaysia",
    "Peru": "Peru", "Qatar": "Qatar", "Tsjechië": "Czech Republic", "Thailand": "Thailand", "Turkije": "Turkey",
    "Verenigde Arabische Emiraten": "United Arab Emirates", "Verenigde Staten": "United States",
    "Verenigd Koninkrijk": "United Kingdom", "Luxemburg": "Luxembourg", "Singapore": "Singapore",
    "Nederland": "Netherlands", "Frankrijk": "France", "Zwitserland": "Switzerland", "Japan": "Japan",
    "Canada": "Canada", "Australië": "Australia", "Hong Kong": "Hong Kong",
    "Europese Unie": "European Union", "Rusland": "Russia", "--": "Unknown",
}
ASSET_CLASS_NL_EN = {"aandelen": "EQUITY", "contanten": "CASH", "geldmarkt": "CASH",
                     "liquiditeiten": "CASH", "cash collateral and margins": "CASH",
                     "money market": "CASH", "fx": "FX", "futures": "FUTURE", "derivaten": "FUTURE"}

In [ ]:
# ---- master-bestand (replicatie voor SPDR en Fidelity) ----
def load_master():
    if not MASTER.exists():
        return pd.DataFrame(columns=["uitgever", "replicatie", "ter"])
    m = pd.read_csv(MASTER, dtype=str, sep=None, engine="python")
    m.columns = [c.strip().lower() for c in m.columns]
    return m.set_index("isin")


@dataclass
class Parsed:
    path: Path
    uitgever: str
    meta: dict
    df: pd.DataFrame
    notes: list = field(default_factory=list)      # waarschuwingen uit de parser (bv. onbekende labels)
    checks: list = field(default_factory=list)     # (niveau, controle, melding)

    @property
    def status(self):
        lv = {c[0] for c in self.checks}
        return "FAIL" if "FAIL" in lv else "WARN" if "WARN" in lv else "OK"

In [ ]:
# ---- parsers: één per uitgever, elk geeft een Parsed terug met df in het schema ----
def _blank_frame(n):
    return pd.DataFrame({c: [None] * n for c in SCHEMA[5:] + EXTRA[1:4]})


def parse_amundi(path, xl):
    raw = xl.parse("Fund Holdings", header=None, dtype=object)
    h = find_header_row(raw, ["ISIN_ETF", "ISIN code", "Name", "Weight"])
    info, snap = {}, None
    for i in range(h):
        a, b = clean(raw.iat[i, 0]), clean(raw.iat[i, 1])
        if a and b:
            info[a.lower()] = b
        if a and snap is None and (m := re.search(r"as at\s+(\d{1,2}/\d{1,2}/\d{4})", a, re.I)):
            snap = parse_date(m[1])
    d = table_below(raw, h)
    d = d[d["Asset class"].map(clean).notna() & d["ISIN_ETF"].map(clean).notna()].reset_index(drop=True)
    df = _blank_frame(len(d))
    df["holding_isin"] = d["ISIN code"].map(clean)
    df["holding_naam"] = d["Name"].map(clean)
    df["asset_class"] = d["Asset class"].map(clean).str.upper()
    df["valuta"] = d["Currency"].map(clean)
    df["gewicht_pct"] = d["Weight"].map(to_float) * 100          # fractie -> procentpunten
    df["sector"] = d["Sector"].map(clean)
    df["land"] = d["Country"].map(clean)
    meta = dict(etf_isin=info.get("isin code"), isin_inhoud=info.get("isin code"),
                etf_naam=info.get("name of the fund"), bron_type="fund", snapshot_date=snap,
                replicatie=norm_replication(info.get("replication method")))
    return Parsed(path, "Amundi", meta, df)


def parse_ishares(path, xl):
    raw = xl.parse("Posities", header=None, dtype=object)
    h = find_header_row(raw, ["Beurscode emittent", "Naam", "Weging (%)"])
    info = {}
    for i in range(h):
        a, b = clean(raw.iat[i, 0]), clean(raw.iat[i, 1])
        if a and b:
            info[a.lower()] = b
    # ISIN en productstructuur staan op de sheet 'Overzicht' (niet op 'Posities')
    ov = xl.parse("Overzicht", header=None, dtype=object) if "Overzicht" in xl.sheet_names else pd.DataFrame()
    ovm = {clean(a): clean(b) for a, b in zip(ov.get(0, []), ov.get(1, [])) if clean(a)}
    d = table_below(raw, h)
    d = d[d["Naam"].map(clean).notna()].reset_index(drop=True)
    notes = []

    def mapped(series, table, label):
        out = series.map(clean)
        miss = sorted({v for v in out.dropna().unique() if v not in table})
        if miss:
            notes.append(f"{label}: geen vertaling voor {miss}; origineel behouden")
        return out.map(lambda v: table.get(v, v) if v else None)

    df = _blank_frame(len(d))
    df["holding_naam"] = d["Naam"].map(clean)
    df["holding_ticker"] = d["Beurscode emittent"].map(clean)
    df["beurs"] = d["Beurs"].map(clean)
    df["valuta"] = d["Beursvaluta"].map(clean)
    df["gewicht_pct"] = d["Weging (%)"].map(to_float)
    df["sector"] = mapped(d["Sector"], SECTOR_NL_EN, "Sector")
    df["land"] = mapped(d["Locatie"], COUNTRY_NL_EN, "Land")
    ac = d["Beleggingscategorie"].map(clean)
    unk = sorted({v for v in ac.dropna().unique() if v.lower() not in ASSET_CLASS_NL_EN})
    if unk:
        notes.append(f"Beleggingscategorie: onbekend {unk}; hoofdletters behouden")
    df["asset_class"] = ac.map(lambda v: ASSET_CLASS_NL_EN.get(v.lower(), v.upper()) if v else None)
    # iShares-ETF's als positie -> fonds-in-fonds (dubbeltelling bij look-through vermijden)
    df["fonds_in_fonds"] = df["holding_naam"].str.upper().str.match(r"^ISHARES\b")
    exp = to_float(info.get("number of securities"))
    meta = dict(etf_isin=ovm.get("ISIN"), isin_inhoud=ovm.get("ISIN"), etf_naam=clean(raw.iat[1, 0]),
                bron_type="fund", snapshot_date=parse_date(info.get("fund holdings as of")),
                replicatie=norm_replication(ovm.get("Productstructuur")),
                expected_equity_rows=None if np.isnan(exp) else int(exp))
    return Parsed(path, "iShares", meta, df, notes)


def parse_spdr(path, xl):
    raw = xl.parse("holdings", header=None, dtype=object)
    h = find_header_row(raw, ["ISIN", "Security Name", "Percent of Fund"])
    info = {}
    for i in range(h):
        a, b = clean(raw.iat[i, 0]), clean(raw.iat[i, 1])
        if a and b:
            info[a.rstrip(":").strip().lower()] = b
    d = table_below(raw, h)
    d = d[d["Security Name"].map(clean).notna()].reset_index(drop=True)   # laat de disclaimer-footer vallen
    df = _blank_frame(len(d))
    isin = d["ISIN"].map(clean)
    unassigned = isin.isna() | (isin.str.lower() == "unassigned")
    df["holding_isin"] = isin.where(~unassigned, None)
    df["holding_naam"] = d["Security Name"].map(clean)
    df["valuta"] = d["Currency"].map(clean)
    df["gewicht_pct"] = d["Percent of Fund"].map(to_float)                 # '-' -> NaN
    sec = d["Sector Classification"].map(clean)
    df["sector"] = sec.where(sec.str.lower() != "unassigned", None)
    land = d["Trade Country Name"].map(clean)
    df["land"] = land.where(land.str.lower() != "none", None)
    price = d["Local Price"].map(to_float)
    # geen asset-classkolom: ISIN 'Unassigned' met koers 1,0 = cash, anders future
    df["asset_class"] = np.where(~unassigned, "EQUITY", np.where((price - 1).abs() < 1e-9, "CASH", "FUTURE"))
    meta = dict(etf_isin=info.get("isin"), isin_inhoud=info.get("isin"), etf_naam=info.get("fund name"),
                bron_type="fund", snapshot_date=parse_date(info.get("holdings as of")), replicatie=None)
    return Parsed(path, "SPDR", meta, df)


def parse_fidelity(path, xl):
    sheet = next(s for s in xl.sheet_names if ISIN_RE.match(s))
    raw = xl.parse(sheet, header=None, dtype=object)
    h = find_header_row(raw, ["ISIN", "Naam", "Weging (%)"])
    title = clean(raw.iat[0, 0]) or ""
    snap = parse_date(raw.iat[1, 1])
    d = table_below(raw, h)
    d = d[d["Naam"].map(clean).notna()].reset_index(drop=True)
    # fonds vs index: de titel van het fonds bevat 'ETF', die van de index niet
    bron = "fund" if "ETF" in title.upper() else "index"
    notes = []
    hint = "index" if "index" in path.name.lower() else "fund" if "fund" in path.name.lower() else None
    if hint and hint != bron:
        notes.append(f"bestandsnaam wijst op '{hint}', titel op '{bron}'")
    df = _blank_frame(len(d))
    df["holding_isin"] = d["ISIN"].map(clean)
    df["holding_naam"] = d["Naam"].map(clean)
    df["gewicht_pct"] = d["Weging (%)"].map(to_float)
    # zonder ISIN = futures (bv. 'SP500 MIC EMIN FUTDEC26'), de rest aandelen
    nofut = df["holding_isin"].isna()
    df["asset_class"] = np.where(~nofut, "EQUITY",
                                 np.where(df["holding_naam"].str.upper().str.contains("FUT", na=False), "FUTURE", "OTHER"))
    meta = dict(etf_isin=sheet, isin_inhoud=sheet, etf_naam=title, bron_type=bron, snapshot_date=snap, replicatie=None)
    return Parsed(path, "Fidelity", meta, df, notes)


PARSERS = {"Amundi": parse_amundi, "iShares": parse_ishares, "SPDR": parse_spdr, "Fidelity": parse_fidelity}


def detect_issuer(xl):
    names = xl.sheet_names
    head = lambda s, n=25: xl.parse(s, header=None, dtype=object, nrows=n)
    if "Fund Holdings" in names and has_text(head("Fund Holdings"), "Amundi"):
        return "Amundi"
    if "Posities" in names:
        try:
            find_header_row(head("Posities", 20), ["Beurscode emittent", "Naam", "Weging (%)"])
            return "iShares"
        except ValueError:
            pass
    if "holdings" in names and has_text(head("holdings", 10), "Fund Name:"):
        return "SPDR"
    for s in names:
        if ISIN_RE.match(s):
            try:
                find_header_row(head(s, 10), ["ISIN", "Naam", "Weging (%)"])
                return "Fidelity"
            except ValueError:
                pass
    raise ValueError("Uitgever niet herkend (geen Amundi/iShares/SPDR/Fidelity-structuur)")

In [ ]:
# ---- afwerking + validatie ----
def finalize(p, master):
    m, df = p.meta, p.df
    if m["bron_type"] == "fund" and not m.get("replicatie") and m["etf_isin"] in master.index:
        m["replicatie"] = norm_replication(master.at[m["etf_isin"], "replicatie"])
    df["snapshot_date"] = m["snapshot_date"]
    df["etf_isin"], df["etf_naam"], df["bron_type"] = m["etf_isin"], m["etf_naam"], m["bron_type"]
    df["replicatie"] = m["replicatie"] if m["bron_type"] == "fund" else None
    df["uitgever"], df["bronbestand"] = p.uitgever, p.path.name
    df["fonds_in_fonds"] = df["fonds_in_fonds"].astype("boolean").fillna(False).astype(bool)
    p.df = df[SCHEMA + EXTRA]


def validate(p):
    m, df, add = p.meta, p.df, (lambda lv, name, msg: p.checks.append((lv, name, msg)))
    add("OK" if m.get("snapshot_date") else "FAIL", "peildatum", str(m.get("snapshot_date")))

    fn = ISIN_FILE_RE.search(p.path.name)
    inh = m.get("isin_inhoud")
    if not inh or not isin_ok(inh):
        add("FAIL", "etf_isin", f"ISIN in bestand ontbreekt of ongeldig: {inh!r}")
    elif fn and fn[0] != inh:
        add("FAIL", "etf_isin", f"bestandsnaam {fn[0]} <> inhoud {inh}")
    else:
        add("OK", "etf_isin", inh)

    n, exp = len(df), m.get("expected_equity_rows")
    if n == 0:
        add("FAIL", "aantal_rijen", "geen posities gelezen")
    elif exp is not None and int(df["asset_class"].eq("EQUITY").sum()) != exp:
        n_equity = int(df["asset_class"].eq("EQUITY").sum())
        add("FAIL", "aantal_rijen", f"{n_equity} aandelen gelezen, kop meldt {exp}; {n} regels totaal")
    else:
        n_equity = int(df["asset_class"].eq("EQUITY").sum())
        add("OK", "aantal_rijen", f"{n_equity} aandelen (= kop), {n} regels totaal" if exp is not None else f"{n} regels")

    tot = df["gewicht_pct"].sum(skipna=True)
    tol = WEIGHT_TOL[p.uitgever]
    add("OK" if abs(tot - 100) <= tol else "FAIL", "gewichtensom", f"{tot:.2f}% (tolerantie ±{tol})")

    keys = pd.Series(index=df.index, dtype=object)
    has_isin = df["holding_isin"].map(clean).notna()
    keys.loc[has_isin] = "ISIN|" + df.loc[has_isin, "holding_isin"].map(clean)
    ticker = df["holding_ticker"].map(clean)
    usable_ticker = ticker.notna() & ~ticker.str.upper().isin({"--", "-", "N/A"})
    fallback = ~has_isin & df["asset_class"].eq("EQUITY") & usable_ticker & df["holding_naam"].map(clean).notna()
    keys.loc[fallback] = [
        (name.casefold(), symbol.casefold(), (venue or "").casefold())
        for name, symbol, venue in zip(df.loc[fallback, "holding_naam"], ticker.loc[fallback], df.loc[fallback, "beurs"])
    ]
    dups = sorted(map(str, keys.dropna().loc[lambda values: values.duplicated(keep=False)].unique()))
    add("FAIL" if dups else "OK", "dubbele_holdings", f"{len(dups)} dubbel: {dups[:5]}" if dups else "geen")
    unchecked = int((~has_isin & df["asset_class"].eq("EQUITY") & ~usable_ticker).sum())
    if unchecked:
        add("WARN", "dubbele_controle_onvolledig", f"{unchecked} aandelen zonder ISIN of bruikbare ticker")

    bad = [i for i in df["holding_isin"].dropna().unique() if not isin_ok(i)]
    if bad:
        add("WARN", "holding_isin_formaat", f"{len(bad)} ongeldig (formaat/controlecijfer): {bad[:5]}")
    if (df["gewicht_pct"] < 0).any():
        add("WARN", "negatief_gewicht", f"{int((df['gewicht_pct'] < 0).sum())} rijen")
    nw = int(df["gewicht_pct"].isna().sum())
    if nw:
        add("WARN", "zonder_gewicht", f"{nw} rijen (cash/futures zonder gewicht)")
    ne = df.loc[df["asset_class"] != "EQUITY", "asset_class"].value_counts().to_dict()
    if ne:
        add("OK", "niet_aandelen", str(ne))
    if m["bron_type"] == "fund" and not m.get("replicatie"):
        add("WARN", "replicatie", "onbekend: voeg ISIN toe aan etf_master.csv")
    nff = int(df["fonds_in_fonds"].sum())
    if nff:
        add("OK", "fonds_in_fonds", f"{nff} posities gemarkeerd, {df.loc[df['fonds_in_fonds'], 'gewicht_pct'].sum():.2f}%")
    for note in p.notes:
        add("WARN", "parser", note)


def load_all():
    master = load_master()
    parsed, errors = [], []
    for path in sorted(INBOX.glob("*.xlsx")):
        if path.name.startswith("~$"):
            continue
        try:
            with pd.ExcelFile(path) as xl:
                p = PARSERS[detect_issuer(xl)](path, xl)
            finalize(p, master)
            validate(p)
            parsed.append(p)
        except Exception as e:
            errors.append((path.name, f"{type(e).__name__}: {e}"))
    return parsed, errors


def report(parsed, errors):
    rows = []
    for p in parsed:
        pick = lambda lv: " | ".join(f"{n}: {msg}" for l, n, msg in p.checks if l == lv)
        rows.append(dict(bestand=p.path.name, uitgever=p.uitgever, etf_isin=p.meta["etf_isin"],
                         bron_type=p.meta["bron_type"], peildatum=p.meta["snapshot_date"], rijen=len(p.df),
                         gewichtensom=round(p.df["gewicht_pct"].sum(), 2), status=p.status,
                         fail=pick("FAIL"), warn=pick("WARN")))
    for name, msg in errors:
        rows.append(dict(bestand=name, status="FAIL", fail=msg))
    return pd.DataFrame(rows)

In [ ]:
# ---- uitvoeren ----
INBOX.mkdir(exist_ok=True)
OUTPUT.mkdir(exist_ok=True)

found = sorted(p.name for p in INBOX.glob("*.xlsx"))
print(f"pandas {pd.__version__} | werkmap: {BASE}")
print(f"inbox: {INBOX} -> {len(found)} .xlsx-bestand(en): {found}")
if not found:
    print("!! Geen .xlsx in inbox/. Draait het notebook vanuit een andere map dan waar inbox/ staat? "
          "Zet BASE bovenaan op de juiste map.")

parsed, errors = load_all()
rep = report(parsed, errors)
display(rep)

use = [p for p in parsed if p.status != "FAIL" or INCLUDE_FAILED]
skipped = [p.path.name for p in parsed if p not in use] + [n for n, _ in errors]
if not use:
    print("\n!! Niets weggeschreven: geen enkel bestand is geslaagd (zie status/fail hierboven).")
if use:
    out = pd.concat([p.df for p in use], ignore_index=True)
    out.to_csv(OUTPUT / "holdings_long.csv", index=False, encoding="utf-8")
    print(f"\nholdings_long.csv: {len(out)} rijen uit {len(use)} bestanden -> {OUTPUT / 'holdings_long.csv'}")
rep.to_csv(OUTPUT / "validation_report.csv", index=False, encoding="utf-8")
if skipped:
    print("NIET opgenomen (FAIL):", skipped)